# AH-1S — Komut curriculum'u (PPO sıfırdan)

Mentorun planı (22 Eylül 2026): **teacher / student yok**. PPO rastgele ağırlıklarla başlar, helikopter
**300 ft / 15 ft/s**'de başlar, kısa episode'larda küçük komutlar verilir: önce **±5° heading**, öğrenince
**±10°**, ... sonra **hız**, sonra **irtifa**. Rüzgâr yok.

| Dosya | Ne yapar |
|---|---|
| `helicopter_env_command.py` | Env: havada başlatma, Δheading/Δhız/Δirtifa komutları, reward, başarı |
| `command_curriculum.py` | Seviyeler (H1 ±5° → H2 ±10° → … → V1–V3 hız → A1–A4 irtifa → C1, R1) |
| `train_command_curriculum.py` | PPO eğitimi + otomatik seviye atlama (başarı ≥ %80) |
| `evaluate_command_policy.py` | Eğitilmiş modelin basamak cevabı (step response) testi |
| `diagnose_command_env.py` | Eğitimsiz sağlık kontrolü (başlatma, sıfır action, açık-döngü tepkiler) |

Sıra: **1 → 2 → 3 → 4**, sonra 5–7 ile sonuçlara bak. Eğitmeden denemek için 6. bölümde repodaki hazır
modelleri (`models_command_curriculum/`) kullanabilirsin.

## 1. Kurulum

In [ ]:
REPO_URL = "https://github.com/Ertovaski7/ah1s-rl-project.git"
BRANCH = "main"
REPO_DIR = "/content/ah1s-rl-project"

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip -q install -r requirements.txt

## 2. Google Drive (önerilir)

Colab oturumu kopunca `/content` silinir. Modeller ve ilerleme dosyaları Drive'a yazılırsa eğitime kaldığın
yerden devam edebilirsin (`--resume`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = "/content/drive/MyDrive/ah1s_runs/komut_curriculum"   # Drive istemezsen: RUN_DIR = "runs/komut_curriculum"
print(RUN_DIR)

## 3. Sağlık kontrolü (eğitim yok, ~1 dk)

Beklenen: `[A]` satırlarında `mode=teleport`, rpm ≈ 320; `[C]`'de rudder +0.5 → yaw rate ≈ −4°/s
(pozitif rudder burnu sola çevirir), collective +0.5 → dikey hız ≈ +4 ft/s.

In [ ]:
%run diagnose_command_env.py

## 4. Eğitim

* PPO **rastgele ağırlıklarla** başlar (teacher yok). Seviye: H1 (±5°).
* Son 100 episode'un başarısı ≥ %80 olunca model `models/level_XX_<ad>.zip` olarak kaydedilir ve
  **aynı ağ** bir sonraki seviyeye geçer.
* Her ~100 bin adımda `models/latest.zip` + `curriculum_state.json` kaydedilir.
* Durdurmak için hücreyi durdur (■). Devam etmek için alttaki `--resume` hücresi.

Episode başarısı: her komuttan sonraki pencerenin son 10 s'si boyunca |heading hatası| ≤ 1.5°,
|hız hatası| ≤ 1.5 ft/s, |irtifa hatası| ≤ 10 ft, güvenlik ihlali yok **ve** komut verilmeyen eksen
pencere boyunca sınır içinde (heading 5°, hız 4 ft/s, irtifa 25 ft — ör. dönüşte hızı kaçırma).
Varsayılan reward/başarı ayarı `v2` (yumuşak kumanda). İlk koşunun ayarı için `--env-config v1`.

Beklenen süre (Claude'un 2 CPU'lu ortamında): H1 ~2–3 dk; tüm 15 seviye ~40–60 dk.
Colab'ın CPU'su daha yavaş olabilir.

In [ ]:
%run train_command_curriculum.py --out {RUN_DIR} --total-steps 6000000

In [ ]:
# Oturum koptuysa / durdurduysan kaldığın yerden devam:
# %run train_command_curriculum.py --out {RUN_DIR} --total-steps 6000000 --resume

## 5. İlerleme grafiği

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f"{RUN_DIR}/progress.csv")
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(df.timesteps, df.success_rate * 100, color="#2a78d6", lw=2)
ax[0].set_ylabel("başarı (son 100 episode) [%]"); ax[0].set_xlabel("adım"); ax[0].grid(alpha=.3)
for lv, g in df.groupby("level", sort=False):
    ax[0].axvline(g.timesteps.iloc[0], color="#9a9994", lw=.8, ls=":")
    ax[0].text(g.timesteps.iloc[0], 102, lv, fontsize=8)
ax[1].plot(df.timesteps, df.final_abs_heading_err, color="#eb6834", lw=2)
ax[1].set_ylabel("komut sonu |heading hatası| [°]"); ax[1].set_xlabel("adım"); ax[1].grid(alpha=.3)
plt.show()
df.groupby("level", sort=False).agg(adim=("timesteps", "max"), dakika=("wall_min", "max"),
                                    basari=("success_rate", "last"))

## 6. Değerlendirme (step response)

Her test tek komut (t = 5 s), 60 s uçuş, deterministik policy. `--zero-baseline` hiç kumanda
oynatmayan (a = 0) durumla karşılaştırır — ajanın gerçekten iş yaptığını gösterir.
Eğitilmemiş büyüklükleri de dene (ör. H1 modeline ±10°).

In [ ]:
%run evaluate_command_policy.py --model {RUN_DIR}/models/level_00_H1.zip --level H1 --zero-baseline

In [ ]:
# Eğitmeden: repodaki hazır modeller (Claude'un koşusu, reward v2). İlk seviye ve son seviye:
%run evaluate_command_policy.py --model models_command_curriculum/v2_level_00_H1.zip --level H1 --zero-baseline
%run evaluate_command_policy.py --model models_command_curriculum/v2_level_14_R1.zip --level C1

In [ ]:
# Son model, büyük dönüşler ve farklı komutlar; eğitimde görülmemiş başlangıç (800 ft, 22 ft/s):
%run evaluate_command_policy.py --model {RUN_DIR}/models/latest.zip --commands heading:+30 heading:-90 heading:+180 speed:+5 speed:-5
%run evaluate_command_policy.py --model {RUN_DIR}/models/latest.zip --commands heading:+90 altitude:-80 --start-alt 800 --start-speed 22

## 7. Görev demosu — tek uçuşta art arda komutlar

Arayüzün yapacağı gibi: aynı FDM'de, sırayla Δheading / Δhız / Δirtifa komutları (`zaman:eksen:değer`).

In [ ]:
%run evaluate_command_policy.py --model {RUN_DIR}/models/latest.zip --mission 5:heading:+90 40:speed:+8 75:altitude:+100 110:heading:-45,altitude:-60 145:speed:-10 180:heading:+180